In [34]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path

In [35]:
def process_all_pdf(pdf_directory):
    
    all_documents=[]
    pdf_dir=Path(pdf_directory) #setting the path(converting the input into a Path object)(instantiation)
    
    pdf_files=list(pdf_dir.glob("**/*.pdf"))# creating a list of file names, glob() is a method in Path
    #pdf_files is now a list of Path objects, one for each PDF found (e.g., [Path('../data/doc1.pdf'), Path('../data/subdir/doc2.pdf')]).
    
    print(f"found {len(pdf_files)} pdf files for processing")
    
    
    
    try:
        for pdf_file in pdf_files:
            
            #used str to convert Path object  back to string
            loader=PyMuPDFLoader(str(pdf_file))#Instantiation
            #documents is a list where one Document object for each page on the pdf
            documents=loader.load()   #load() method 
            
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']="pdf"
            
            print(f"meta data added for all pages")    
            all_documents.extend(documents)
            print(f'loaded:{len(documents)} pages of pdf named: {pdf_file.name}')
    
    except Exception as e:
        print(f"Error:{e}")
    print(f"loaded:{len(all_documents)} documents")
    return all_documents

all_pdf_documents=process_all_pdf('../data')

found 2 pdf files for processing
meta data added for all pages
loaded:8 pages of pdf named: Hanseg oar.pdf
meta data added for all pages
loaded:9 pages of pdf named: SSRepL-ADHD.pdf
loaded:17 documents


In [36]:
print(all_pdf_documents)

[Document(metadata={'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2024-08-05T11:07:15+00:00', 'source': '..\\data\\pdf_files\\Hanseg oar.pdf', 'file_path': '..\\data\\pdf_files\\Hanseg oar.pdf', 'total_pages': 8, 'format': 'PDF 1.7', 'title': 'HaN-Seg: The head and neck organ-at-risk CT and MR segmentation challenge', 'author': 'Gašper Podobnik', 'subject': 'Radiotherapy and Oncology, 198 (2024) 110410. doi:10.1016/j.radonc.2024.110410', 'keywords': 'Computational challenge,Segmentation,Deep learning,Organs-at-risk,Computed tomography,Magnetic resonance,Radiotherapy,Head and neck cancer', 'moddate': '2024-08-05T11:08:40+00:00', 'trapped': '', 'modDate': 'D:20240805110840Z', 'creationDate': 'D:20240805110715Z', 'page': 0, 'source_file': 'Hanseg oar.pdf', 'file_type': 'pdf'}, page_content='Radiotherapy and Oncology 198 (2024) 110410\nAvailable online 23 June 2024\n0167-8140/© 2024 The Authors. Published by Elsevier B.V. This is an open access ar

Text splitting(chunking)

In [37]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split the pdf into smaller chunks for addresing the context window size"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,#Measure my chunks by counting the number of characters. for real project we need tokens here
        separators=["\n\n","\n"," ",""]#split at paragraphs,at new lines, after each word, after each charecter. this is the order. 
    )
    split_docs=text_splitter.split_documents(documents)
    #print(type(split_docs))
    print(f"splitted {len(documents)} documents into {len(split_docs)}chunks")
    
    if split_docs:
        print("\n example chunk")
        print(f"page contnt: {split_docs[0].page_content[:200]}")
        print(f"the metadata: {split_docs[0].metadata}")
    return split_docs

In [38]:
#mine using tiktoken, instead using charecter as 1000 we use tokens, in real world, llm has token limit not charecter limit

def split_documents_by_tokens(documents, chunk_size=1000, chunk_overlap=200):
    
    # We use .from_tiktoken_encoder instead of the standard class ()
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        model_name="gpt-4", # Or "gpt-3.5-turbo"
        chunk_size=chunk_size,   # NOW this means 1000 TOKENS
        chunk_overlap=chunk_overlap
    )

In [39]:
chunks=split_documents(all_pdf_documents)#List of documents
print(chunks
      )

splitted 17 documents into 122chunks

 example chunk
page contnt: Radiotherapy and Oncology 198 (2024) 110410
Available online 23 June 2024
0167-8140/© 2024 The Authors. Published by Elsevier B.V. This is an open access article under the CC BY license (http://creati
the metadata: {'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2024-08-05T11:07:15+00:00', 'source': '..\\data\\pdf_files\\Hanseg oar.pdf', 'file_path': '..\\data\\pdf_files\\Hanseg oar.pdf', 'total_pages': 8, 'format': 'PDF 1.7', 'title': 'HaN-Seg: The head and neck organ-at-risk CT and MR segmentation challenge', 'author': 'Gašper Podobnik', 'subject': 'Radiotherapy and Oncology, 198 (2024) 110410. doi:10.1016/j.radonc.2024.110410', 'keywords': 'Computational challenge,Segmentation,Deep learning,Organs-at-risk,Computed tomography,Magnetic resonance,Radiotherapy,Head and neck cancer', 'moddate': '2024-08-05T11:08:40+00:00', 'trapped': '', 'modDate': 'D:20240805110840Z', 'creationDat

Embedding & Vector db

In [40]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings #Allows us to configure the database (e.g., "Save the data to my hard drive permanently" vs. "Keep it in RAM only").
#Universally Unique Identifier
import uuid # give unique id to each chunk
from typing import List, Tuple, Any, Dict
from sklearn.metrics.pairwise import cosine_similarity #ou use Cosine Similarity to compare the Question Vector vs. your Document Vectors.

In [ ]:
class EmbeddingManager:
    """
    initializing the embedding manager
    
    Args:
    model name: hugging face  model name for sentence embedding"""
    def __init__(self,model_name:str = "all-MiniLM-L6-v2"):
        """When you create a new instance(object) of a class, Python automatically runs __init__ immediately.
        It ensures the object is "born" with all the data it needs to function.
        
        Without __init__: You would have to write 3 lines of code every time you wanted to use the manager, 
        and if you forgot one, the program would crash.
        """
        # 1. Save the model name to 'self' so other functions can use it later.
        self.model_name=model_name#Save the name immediately
        
        # 2. Create a placeholder variable for the model. 
        # It is 'None' right now because we haven't loaded it yet.
        self.model=None
        
        # 3. Immediately call the internal function to download and load the model.
        # We do this here so the embedding_manager is ready to use the moment it's created.
        self._load_model()# Automatically load the model when we start
        """
        self.variable (With self): This is a permanent tattoo on the object.
        It stays with the object forever, and any other function in the class can read it.
        """
    def _load_model(self):
        """ Load sentence transformer model"""
        try:
        
            print(f"loading the embedding model{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            
            dim=self.model.get_sentence_embedding_dimension()
            
            print(f"loaded the model{self.model_name} the embedding dimension : {dim}")
            
        except Exception as e:
            print(f"error in loading model {self.model_name}:{e}")
            
            # 5. Stop the program. We cannot continue without a model.
            raise
        
        
    def generate_embeddings(self,text:List[str])-> np.ndarray:
        """ Generate embedding for a list text
        Args:
        list of texts to embed
        
        Returns:
        numpy array embeddings with shape(len(text),embed_dim)
        """
        
        # 1. Safety Guard:
        # Check if the model actually loaded successfully before trying to use it.
        if not self.model:
            raise ValueError("Model not loaded")
        
        
        # 2. The Core Task:
        # .encode() is the library function that does the math.
        # show_progress_bar=True gives a visual indicator for large lists.
        print(f"generating embedding for{len(text)} texts")
        embeddings=self.model.encode(text,show_progress_bar=True)
        print(f"Generated embeddings with shape{embeddings.shape}")
        return embeddings
        
        


In [ ]:
#initialise the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

"""
Where did self go? Python passes it automatically! When you call      manager._load_model()      , Python silently translates it to:

                         EmbeddingManager._load_model(manager)

It takes the object on the left of the dot (manager) and passes it as the first argument (self).
"""

loading the embedding modelall-MiniLM-L6-v2


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 1959087f-d4d4-4051-bff9-74836e5ac6ab)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


loaded the modelall-MiniLM-L6-v2 the embedding dimension : 384


vector store

In [ ]:
class VectorStore:
    """
    manages document embeddings in Chromadb vectorstore
    
    """
    def __init__(self,collection_name:str="pdf documents",persist_directory:str="../data/vector_store"):
        """
        Initialize the VectorStore
        
        Args:
        collection_name: Name of the chroma db collection
        persistant directery: directory to persist the directory
        
        """
        
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self.initialize_store()
    
    def initialize_store(self):
        """ here we initialize the client and  collection
        """
        try:
            #create a client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)#creating a client having reference to the vector store
            
            #get or create a collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"Pdf document embedding for for rag"}
                
            )
            print(f"vector store initialized. Collection:{self.collection_name}")
            print(f"existing documents in collection:{self.collection.count()}")
        
        except Exception as e:
            print(f"error initializing vector store:{e}") 
            raise
    
    
            
            
            
        